# **Damage propagation in a U-bolt:**
In the following parts we will focus on the problem of damage propagation over time. In the following lines of code we will upload the data computed by solving a **F.O.M.**.
This high fidelity solutions are computed starting from the **Fisher-KPP** equation:
<br><br><br>
$$
\begin{cases}
\frac{\partial u}{\partial t} - D \Delta u = r u(1-u) & \text{in } \Omega \times (0,T], \\
\nabla u \cdot \mathbf{n} = 0 & \text{on } \partial \Omega \times (0,T], \\
u(\cdot, 0) = g_\delta & \text{in } \Omega,
\end{cases}
$$
<br><br><br>
where $D > 0$ and $r > 0$ are the **diffusion and reaction coefficients**, respectively, whereas $\delta > 0$ is a parameter describing the location of **the initial damage**. While, the solution $u : \Omega \times [0, T] \rightarrow [0, 1]$ indicates the level of damage:

*   $u$ equal to $1$ indicates maximum damage
*   $u$ equal to $0$ indicates no damage



 Then for $g_\delta$ we have that:


$$
g_\delta(x,y) :=
\begin{cases}
1 & \text{if } |x - \delta| < 0.1 \text{ and } y \geq 4, \\
0 & \text{otherwise}.
\end{cases}
$$


For the sake of our analysis, we parametrize the model coefficients as


$$
D = 10^{\mu_1}, \quad r = 10^{\mu_2 + 2}, \quad \delta = \mu_3 + 1,
$$


where $\mu = [\mu_1, \mu_2, \mu_3]^\top$ varies in $[0,1]^3$. The final time is set to $T = 0.02$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
     from dlroms import*
except:
     !pip install --no-deps git+https://github.com/NicolaRFranco/dlroms.git
     from dlroms import*

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd
import time
from IPython.display import clear_output as clc
import torch
from torch import tanh
from torch.optim import Adam

In [ ]:
# Fancy colormap for plotting
from matplotlib.colors import ListedColormap
import matplotlib as mpl
jet = mpl.colormaps['jet']
jet_colors = jet(np.linspace(0, 1, 256))
def modify_jet_to_gray(jet_colors):
    new_colors = jet_colors.copy()
    gray_vals = np.linspace(0.3, 0.7, 85)
    for i in range(85):
        new_colors[i, 0:3] = gray_vals[i]
    return ListedColormap(new_colors)
gray_jet = modify_jet_to_gray(jet_colors)

In [ ]:
# FOM discretization
import gdown
gdown.download(id = "1_4uC_yjWmvaDfAuRt0I1PI3IENE4XYXX", output = "ubolt_mesh.xml")
mesh = fe.loadmesh("ubolt_mesh.xml")
Vh = fe.space(mesh, 'CG', 1)

Downloading...
From (original): https://drive.google.com/uc?id=1_4uC_yjWmvaDfAuRt0I1PI3IENE4XYXX
From (redirected): https://drive.google.com/uc?id=1_4uC_yjWmvaDfAuRt0I1PI3IENE4XYXX&confirm=t&uuid=b17dc845-8160-4512-86b3-83b342abccac
To: /content/ubolt_mesh.xml
100%|██████████| 1.33M/1.33M [00:00<00:00, 103MB/s]

Calling FFC just-in-time (JIT) compiler, this may take some time.



Level 25:FFC:Calling FFC just-in-time (JIT) compiler, this may take some time.
INFO:FFC:Compiling element ffc_element_3801828c0f66b7190a7fd5819465b3d5b34b9149

INFO:FFC:Compiler stage 1: Analyzing element(s)
INFO:FFC:--------------------------------------
INFO:FFC:  
INFO:FFC:Compiler stage 1 finished in 0.00647759 seconds.

INFO:FFC:Compiler stage 2: Computing intermediate representation
INFO:FFC:-------------------------------------------------------
INFO:FFC:  Computing representation of 1 elements
DEBUG:FFC:  Reusing element from cache
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 1 dofmaps
DEBUG:FFC:  Reusing element from cache
INFO:FFC:  Computing representation of 0 coordinate mappings
INFO:FFC:  Computing representation of integrals
INFO:FFC:  Computing representation of forms
INFO:FFC:  
INFO:FFC:Compiler stage 2 finished in 0.405652 seconds.

INFO:FFC:Compiler stage 3: Optimizing intermediate representation
INFO:FFC:---------------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#now we load from the drive the dataset that we have already computed
#load_path = "/content/drive/MyDrive/Colab Notebooks/MODEL ORDER REDUCTION TECHNIQUES/ASSIGNEMENT_2/" #michele
load_path = "/content/drive/MyDrive/Colab Notebooks/ASSIGNEMENT_2/ASSIGNEMENT_2/" #mamma
mu = np.load(load_path + "mu.npy")
ufom = np.load(load_path + "ufom.npy")
clc()
print(f"The shape of the parameters, mu, is: {mu.shape}")
print(f"The shape of the uFom solution is: {ufom.shape}")
print(f"In particular:")
print(f"The number of simulation, Ns, is: {ufom.shape[0]}")
print(f"The number of time step per each simulation, Nt, is: {ufom.shape[1]}")
print(f"The number of mesh nodes, Nh, is: {ufom.shape[2]}")

The shape of the parameters, mu, is: (100, 3)
The shape of the uFom solution is: (100, 41, 6636)
In particular:
The number of simulation, Ns, is: 100
The number of time step per each simulation, Nt, is: 41
The number of mesh nodes, Nh, is: 6636


In [ ]:
#we immediately pass to tensors
mu, ufom = dv.tensor(mu, ufom)

In [ ]:
#we try to see how is the singular value decay even if we know that we will not use POD-GALERKIN
ntraining = 75
ntest = 25
nvalidation = int(ntraining*0.2) #this is used to avoid overfitting wiht the training data
Ns = ufom.shape[0] #the number of simulations
Nt = ufom.shape[1] #the number of time istances
Nh = ufom.shape[2] #the number of nodes
p = mu.shape[1] #the number of parameters
delta_t = 0.0005

In [ ]:
l2 = L2(Vh) #we create the norm  and it expects input like ... x Nh
l2.cuda()
clc()

In [ ]:
#now we define the error function that we will use several times in the code

def error_func(utrue, upred):
  #we must have data written as ... x Nh, otherwise L2 doesn't work
  current_Ns = utrue.shape[0]//Nt #because this will enter also in the training and there Ns is 75
  error =  (l2(utrue-upred).reshape(current_Ns, Nt)/l1Ns, Nt)).mean(axis = -1)
  error = error.mean()
  return error

In [ ]:
#WE CANNOT USE POD + GALERKIN BECAUSE THE PROBLEM IS NON LINEAR AND NON AFFINE AND SO IF WE STILL WANT TO BUILD A ROM WE MUST USE HYPER REDUCTION TECHNIQUES THAT ARE VERY EXPENSIVE

# **Construction of the DL-ROM:**
The **Fisher-KPP** equation is characterized by nonlinear terms and the parameters enter in the equation in a non affine way. So, instead of using **hyper-reduction techniques**, we exploit **neural networks**. In this case the strategy that is adopted is the **DL-ROM**. This is composed by:


*   **encoder**, $\Psi':\mathbb{R}^{N_h}\to\mathbb{R}^{n}$
*   **decoder**, $\Psi:\mathbb{R}^{n}\to\mathbb{R}^{N_{h}}$
*   $\phi:\mathbb{R}^{p}\to\mathbb{R}^{n}$


Basically the **encoder** and the **decoder** play the role of *compressing* and *decompressing* the information. In fact we pass from $N_h$ to $N$, then again from $N$ to $N_h$.
While $\phi$ is used to pass from the parameter domain, $p (+1)$, to the reduced space $N$.

The problem in our case is much more complex because we have to deal also with time. In fact:

*   $u_{\text{fom}}\in\mathbb{R}^{N_s\times N_t\times N_h}$;

*   $\mu\in\mathbb{R}^{N_s\times p}$;

With this approach time is considered as a further paramete, in fact $\mu$ is rewritten in this way:

$$\mu^t\in\mathbb{R}^{N_s\times N_t\times (p+1)}$$ such that
$$\mu^t_{i,j} = [\mu_{i,1},\;\mu_{i,2},\;\mu_{i,3},\;t_j]$$


where we are looking at $i$-th simulated trajectory and the $j$-th time instant.
<br><br><br>
After the training of $\Psi', \Psi, \phi$ we can finally create our **R.O.M.** by saying that:

$$u_{\text{fom}}\approx \Psi(\phi(\boldsymbol{\mu})) = u_{\text{rom}}$$
<br><br><br>
Then we compare the **F.O.M.** with the **DL-ROM** results using the following expression:

$$
\quad
\frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}}
\left[
\frac{1}{N_{\text{test}}} \sum_{j=1}^{N_{\text{test}}}
\frac{\| u_{\text{test}_{i,j}} - \tilde{u}_{\text{test}_{i,j}} \|_{L^2(\Omega)}}
{\| u_{\text{test}_{i,j}} \|_{L^2(\Omega)}}
\right]
$$

where:


*    $u_{test}$ is $u_{fom}$ for the test dataset
*   $\tilde{u}_{\text{test}}$ is $u_{rom}$ for the test dataset


In [ ]:
from scipy.linalg import svd
ufom_DLRom = ufom.reshape(-1, Nh)
ntrain_DLRom = ntraining*Nt
nvalid_DLRom = nvalidation*Nt

t_final = 0.02
times_DLRom = torch.linspace(0, t_final, Nt)
mut_DLRom = torch.zeros(Ns, Nt, p + 1)

# Ciclo for usando solo tensori
for i in range(Ns):
    mut_DLRom[i, :, :p] = mu[i]
    mut_DLRom[i, :, p] = times_DLRom

mut_DLRom = mut_DLRom.reshape(-1, p+1)
mut_DLRom = dv.tensor(mut_DLRom)

/usr/local/lib/python3.11/dist-packages/dlroms/cores.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return ttensor(arrays[0], dtype = self.dtype, device = self.device)


In [ ]:
#rho = lambda x: tanh(0.01*x) ### <-- smooth activation that bounds the dynamics within [-1,1]^n
#rho_e = lambda x: relu(x) -10*relu(-x)
#rho_d = lambda x: relu(x) -0.1*relu(-x)

#latent = 2*(p + 1) + 1
#encoder_DLRom = Dense(Nh, 50, rho_e) + Dense(50, latent, rho)
#decoder_DLRom = Dense(latent, 50, rho_d) + Dense(50, 500, rho_d) + Dense(500, Nh, activation = None)
#ae_DLRom = DFNN(encoder_DLRom + decoder_DLRom)

#ae_DLRom.He() #we prepare wheights and biases for training
#ae_DLRom.cuda()
#ae_DLRom.train(ufom_DLRom, ufom_DLRom, loss = mse(l2), ntrain = ntrain_DLRom, epochs = 1000, error = error_func, notation = '%')
#ae_DLRom.train(ufom_DLRom, ufom_DLRom, ntrain = ntrain_DLRom, epochs = 2000, loss = mse(l2), optim = Adam, lr = 1e-5, batchsize = Nt, error = error_func, notation = '%')
#ae_DLRom.freeze()


In [ ]:
#print("Autoencoder MRE: %s" % num2p(error_func(ufom_DLRom[ntrain_DLRom:], ae_DLRom(ufom_DLRom[ntrain_DLRom:]))))

In [ ]:
rho = lambda x: tanh(0.01*x) ### <-- smooth activation that bounds the dynamics within [-1,1]^n
rho_e = lambda x: relu(x) -10*relu(-x)
rho_d = lambda x: relu(x) -0.1*relu(-x)

latent = 2*(p + 1) + 1
encoder_DLRom = Dense(Nh, 50, rho_e) + Dense(50, latent, rho)
decoder_DLRom = Dense(latent, 50, rho_d) + Dense(50, 500, rho_d) + Dense(500, Nh, activation = None)
ae_DLRom = DFNN(encoder_DLRom + decoder_DLRom)

ae_DLRom.load_state_dict(torch.load(load_path + "autoencoder_DLRom.pt"))
ae_DLRom.cuda()
ae_DLRom.freeze()

In [ ]:
#now we create phi, the newtork that goes from mu to latent
uLatent_DLRom = encoder_DLRom(ufom_DLRom)

phi_DLRom = DFNN(Dense(p + 1, 128, gelu) + Dense(128, 128, gelu) + Dense(128, 128, gelu) + Dense(128, latent, activation = None))

phi_DLRom.He()
phi_DLRom.cuda()
phi_DLRom.train(mut_DLRom, uLatent_DLRom, ntrain = ntrain_DLRom, epochs = 500, loss = mse(euclidean))
#phi_DLRom.train(mut_DLRom, uLatent_DLRom, ntrain = ntrain_DLRom, epochs = 1000, loss = mse(euclidean), lr = 1e-4, optim = Adam, batchsize = Nt)
phi_DLRom.freeze()

		Train		Test
Epoch 500:	6.52e-04	9.12e-03.

>> ETA: 0.28s.

Training complete. Elapsed time: 2 minutes 21.73 seconds.


In [ ]:
#save_path = "/content/drive/MyDrive/Colab Notebooks/MODEL ORDER REDUCTION TECHNIQUES/ASSIGNEMENT_2/" #michele
save_path = "/content/drive/MyDrive/Colab Notebooks/ASSIGNEMENT_2/ASSIGNEMENT_2/" #mamma

In [ ]:
#we save our autoencoder to use it in other notebooks
torch.save(phi_DLRom.state_dict(), save_path + "phi_DLRom.pt")
np.save(save_path + "latent_DLRom.npy", latent)